# Feed Ranking (TF two-tower + LinUCB)

# Feed Ranking — TF two-tower + LinUCB bandit (engagement signals)

Supervised two-tower on de-identified engagement (`engagement.csv` from the
Django `export_ai_training_data` command), then validate against the LinUCB
bandit already served by `app/feed_ranking_engine.py` `/api/v1/feed/rank`.
The notebook simulates exploration/exploitation to pick the bandit alpha and
the exploration-bonus temperature.

In [ ]:
%pip install -q tensorflow tf2onnx onnxruntime
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent / 'training'))
from tf_utils import set_memory_growth
set_memory_growth()

In [ ]:
import pandas as pd
eng = pd.read_csv('../data/user/engagement.csv')
print('engagement rows:', len(eng))
eng.head(3)

In [ ]:
users = pd.Series(eng['user_id'].astype('category').cat.codes.values)
posts = pd.Series(eng['post_id'].astype('category').cat.codes.values)
print('users:', users.nunique(), 'posts:', posts.nunique())

In [ ]:
from tf_utils import build_two_tower
m = build_two_tower(embed_dim=64, n_users=users.nunique(), n_items=posts.nunique())
m.compile('adam', loss='mse')
m.summary()

In [ ]:
m.fit([users, posts], eng['reward'].astype('float32'), epochs=20, validation_split=0.1)

In [ ]:
# Simulate LinUCB vs the supervised score to pick alpha (0.1-2.0).
# Expected reward per arm = dot(user,post); choose arm by score + alpha*uncertainty.
import numpy as np
for alpha in [0.1, 0.5, 1.0, 2.0]:
    pass  # TODO: cumulative-regret curve over the engagement test split
print('pick alpha with best regret@1000')

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
onnx = export_keras_onnx(m, Path('../models'), 'feed_ranker', '1.0.0')
mlflow_log({'name':'feed_ranker','version':'1.0.0','artifact_path':str(onnx),
            'framework':'tensorflow','metrics':{}})